# CLEF 2026 CheckThat! Task 1 — v5

End-to-end pipeline for source retrieval of scientific web claims.

**Pipeline:**
1. Doc-side augmentation — GPT-4o-mini generates synthetic tweet-style queries per paper (one-time, cached)
2. Query translation — GPT-4o-mini translates DE/FR queries to EN (cached)
3. Iterative hard-negative mining + bi-encoder fine-tune (mE5-large, 2 rounds)
4. First-stage retrieval — FT dense + BM25 RRF → top-100
5. Cross-encoder reranker fine-tune (bge-reranker-v2-m3, hard negatives)
6. Cross-encoder rerank → top-20



# 0) Setup


In [ ]:
!pip install -q transformers datasets sentence-transformers rank-bm25 openai tenacity huggingface_hub

In [ ]:
import os, re, json, time, random, hashlib, math, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample, losses
from datasets import load_dataset, Dataset as HFDataset
from rank_bm25 import BM25Okapi
from tqdm import tqdm
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# === CONFIG ===
# >>>>> PUT YOUR OPENAI KEY HERE <<<<<
OPENAI_API_KEY = ""

OPENAI_MODEL_FAST = "gpt-4o-mini"      # used for translation + doc-aug + listwise rerank
OPENAI_CONCURRENCY = 2                 # parallel OpenAI requests

DENSE_MODEL    = "intfloat/multilingual-e5-large"
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

LANGS  = ["de", "fr", "en"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DS_NAME = "sschellhammer/CT26_Task1_SourceRetrievalForScientificWebClaims"

CACHE_DIR = Path("./v5_cache"); CACHE_DIR.mkdir(exist_ok=True)
MODEL_DIR = Path("./v5_models"); MODEL_DIR.mkdir(exist_ok=True)
OUT_DIR   = Path("./v5_out");    OUT_DIR.mkdir(exist_ok=True)

# Hyperparams
N_SYNTH_TWEETS_PER_PAPER = 2
N_HARD_NEGS              = 5
HARD_NEG_POOL            = 50
N_FT_ROUNDS              = 2
FT_EPOCHS                = 2
FT_BATCH_SIZE            = 32
RERANK_FT_EPOCHS         = 1
RERANK_FT_BATCH          = 16
FIRST_STAGE_TOPK         = 100
CE_RERANK_TOPK           = 20
LLM_RERANK_TOPK          = 10  # Reduced to 10 for speed and efficiency

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
oai = OpenAI(api_key=OPENAI_API_KEY)
print(f"Device: {DEVICE}")
print(f"Cache:  {CACHE_DIR.resolve()}")

# 1) Load data


In [ ]:
from huggingface_hub import login
# Either pass HF token here or run notebook_login() in a separate cell
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    from huggingface_hub import notebook_login
    notebook_login()

In [ ]:
collection_raw = load_dataset(DS_NAME, "collection")["collection"]
collection_keys = np.array(collection_raw["pubkey"])
N_DOCS = len(collection_raw)
print(f"Collection: {N_DOCS} papers")

lang2train = {}
lang2dev   = {}
for lang in LANGS:
    data = load_dataset(DS_NAME, lang)
    lang2train[lang] = data["train"]
    lang2dev[lang]   = data["dev"]
    print(f"  {lang}: {len(data['train'])} train, {len(data['dev'])} dev")

test_data = load_dataset(DS_NAME, name="test")
lang2test = {lang: test_data[lang] for lang in LANGS}
for lang in LANGS:
    print(f"  {lang} test: {len(lang2test[lang])}")

# 2) OpenAI helpers (cached + concurrent + retry)

All OpenAI calls go through a disk cache keyed by `sha256(prompt)`. Re-running the
notebook only pays for new prompts.

In [ ]:
class JsonCache:
    def __init__(self, path: Path):
        self.path = path
        self.data = json.loads(path.read_text()) if path.exists() else {}
    def __contains__(self, k): return k in self.data
    def __getitem__(self, k):  return self.data[k]
    def __setitem__(self, k, v):
        self.data[k] = v
    def flush(self):
        tmp = self.path.with_suffix(".tmp")
        tmp.write_text(json.dumps(self.data))
        tmp.replace(self.path)

def hkey(*parts) -> str:
    return hashlib.sha256("\n--\n".join(parts).encode()).hexdigest()[:24]

@retry(stop=stop_after_attempt(5), wait=wait_exponential(multiplier=1, min=1, max=20))
def oai_chat(messages, model=OPENAI_MODEL_FAST, temperature=0.0, max_tokens=512, response_format=None):
    kwargs = dict(model=model, messages=messages, max_completion_tokens=max_tokens)
    if model != OPENAI_MODEL_FAST:  # only set temperature for models that support it
        kwargs["temperature"] = temperature
    if response_format:
        kwargs["response_format"] = response_format
    r = oai.chat.completions.create(**kwargs)
    return r.choices[0].message.content

def run_concurrent(jobs, fn, cache: JsonCache, desc="OpenAI", concurrency=OPENAI_CONCURRENCY, flush_every=200):
    """jobs: list of (key, payload). fn(payload) -> result. Cached by key."""
    todo = [(k, p) for k, p in jobs if k not in cache]
    print(f"{desc}: {len(jobs)} total, {len(todo)} new")
    if not todo:
        return
    with ThreadPoolExecutor(max_workers=concurrency) as ex:
        futs = {ex.submit(fn, p): k for k, p in todo}
        done = 0
        for fut in tqdm(as_completed(futs), total=len(futs), desc=desc):
            k = futs[fut]
            try:
                cache[k] = fut.result()
            except Exception as e:
                print(f"  fail {k}: {e}")
                cache[k] = None
            done += 1
            if done % flush_every == 0:
                cache.flush()
    cache.flush()

# 3) Load English query translations

Train + dev EN translations are pre-computed in `{train,dev}_en_queries_{lang}.json`
(ordered lists matching the dataset). Test EN translations are produced on-demand
via GPT-4o-mini (cached to disk).

In [ ]:
from torch.serialization import load
TRANSLATE_CACHE = JsonCache(CACHE_DIR / "translations.json")
TRANSLATIONS_DIR = Path(".")  # JSON files live next to the notebook

def load_pretrans(split, lang, expected_len):
    p = TRANSLATIONS_DIR / f"{split}_en_queries_{lang}.json"
    if not p.exists():
        raise FileNotFoundError(f"Missing pre-translated file: {p}")
    arr = json.loads(p.read_text())
    if len(arr) != expected_len:
        raise ValueError(f"{p}: expected {expected_len} translations, got {len(arr)}")
    return arr

def translate_one(payload):
    lang_full = {"de": "German", "fr": "French"}[payload["lang"]]
    prompt = (
        f"Translate the following {lang_full} social-media post to English. "
        "Preserve scientific terminology, drug/compound names, journal names, "
        "study venue names, and author names exactly as they appear. "
        "Output only the English translation, nothing else.\n\n"
        f"Text: {payload['text']}"
    )
    return oai_chat([{"role": "user", "content": prompt}], max_tokens=400).strip()

def translate_test_split(split_data, lang):
    if lang == "en":
        return [ex["text"] for ex in split_data]
    jobs = []
    for ex in split_data:
        k = hkey("translate-v1", lang, ex["text"])
        jobs.append((k, {"lang": lang, "text": ex["text"]}))
    run_concurrent(jobs, translate_one, TRANSLATE_CACHE, desc=f"translate test {lang}")
    return [TRANSLATE_CACHE[k] or ex["text"] for (k, _), ex in zip(jobs, split_data)]

lang2train_en = {}
lang2dev_en   = {}
lang2test_en  = {}
for lang in LANGS:
    lang2train_en[lang] = load_pretrans("train", lang, len(lang2train[lang]))
    lang2dev_en[lang]   = load_pretrans("dev",   lang, len(lang2dev[lang]))
    lang2test_en[lang]  = load_pretrans("test",  lang, len(lang2test[lang]))
    print(f"  {lang}: train={len(lang2train_en[lang])}, dev={len(lang2dev_en[lang])} (loaded from disk), test={len(lang2test_en[lang])}")

# print("\n--- test (translating via GPT-4o-mini) ---")
# for lang in LANGS:
#     lang2test_en[lang] = translate_test_split(lang2test[lang], lang)
# print("Test translations done.")

In [ ]:
print("Saving test translations to disk...")
for lang in LANGS:
    output_path = TRANSLATIONS_DIR / f"test_en_queries_{lang}.json"
    with open(output_path, "w") as f:
        json.dump(lang2test_en[lang], f)
    print(f"  Saved {output_path}")
print("Test translations saved.")

# 4) Doc-side augmentation (doc2query)

For each paper, ask GPT-4o-mini to write `N_SYNTH_TWEETS_PER_PAPER` short tweet-style claims
that someone might post when citing the paper. These are appended to the doc text — the bi-encoder
and BM25 will see vocabulary that bridges the tweet→paper gap.

Cost on 10K papers × 2 tweets at 4o-mini: ~$2.

In [ ]:
DOCAUG_CACHE = JsonCache(Path("/content/v5_cache/doc_synth_tweets.json"))

DOCAUG_SYS = (
    "You write short social-media posts that cite scientific papers. "
    "Given a paper, output a JSON object {\"tweets\": [...]} containing exactly "
    f"{N_SYNTH_TWEETS_PER_PAPER} short tweet-like claims (1-2 sentences each) "
    "that a real user might post when implicitly referencing this paper. "
    "Vary the framing: one may mention the journal/venue, another a key finding, "
    "an author, a specific drug/compound/method. Use natural language, not jargon. "
    "Output ONLY the JSON object, no markdown."
)

def docaug_one(payload):
    user = (
        f"Title: {payload['title']}\n"
        f"Venue: {payload['venue']}\n"
        f"Authors: {payload['authors'][:300]}\n"
        f"Abstract: {payload['abstract'][:1500]}"
    )
    out = oai_chat(
        [{"role": "system", "content": DOCAUG_SYS}, {"role": "user", "content": user}],
        max_tokens=120,
        response_format={"type": "json_object"},
    )
    obj = json.loads(out)
    return obj.get("tweets", [])

# We must ensure jobs is defined to check the cache against it
jobs = []
for ex in collection_raw:
    k = hkey("docaug-v1", str(N_SYNTH_TWEETS_PER_PAPER), ex["title"] or "", (ex["abstract"] or "")[:500])
    jobs.append((k, {
        "title": ex["title"] or "",
        "venue": ex["venue"] or "",
        "authors": ex["authors"] or "",
        "abstract": ex["abstract"] or "",
    }))

In [ ]:
doc_synth = [(DOCAUG_CACHE[k] if k in DOCAUG_CACHE else []) or [] for k, _ in jobs]
print(f"Synth tweets generated for {sum(1 for x in doc_synth if x)} / {len(doc_synth)} docs.")
if any(doc_synth):
    print("Sample:", next(x for x in doc_synth if x))
else:
    print("Sample: (empty)")

# 5) Build augmented collection texts


In [ ]:
def build_doc_text(ex, synth):
    base = f'{ex["title"]}. {ex["venue"]}. {ex["abstract"]}. {ex["authors"]}'
    if synth:
        base += " || " + " ".join(synth)
    return base

collection_texts = [build_doc_text(ex, s) for ex, s in zip(collection_raw, doc_synth)]
collection_texts_plain = [
    f'{ex["title"]}. {ex["venue"]}. {ex["abstract"]}'  # for cross-encoder (no authors / no synth)
    for ex in collection_raw
]
print("Sample doc text (truncated):")
print(collection_texts[0][:600], "...")

# 6) Bi-encoder helpers (mE5-large)


In [ ]:
def load_e5(path=DENSE_MODEL):
    m = SentenceTransformer(path, device=DEVICE)
    m.max_seq_length = 512
    return m

def encode_e5(model, texts, prefix, batch_size=128):
    pref = [f"{prefix}: {t}" for t in texts]
    return model.encode(
        pref, batch_size=batch_size, normalize_embeddings=True,
        convert_to_tensor=True, show_progress_bar=True
    ).cpu()

ft_model = load_e5()

# 7) Initial encode (zero-shot mE5) for round-1 hard-negative mining


In [ ]:
print("Encoding collection (zero-shot)...")
coll_emb = encode_e5(ft_model, collection_texts, prefix="passage", batch_size=128)
print("coll_emb:", coll_emb.shape)

In [ ]:
pubkey_to_idx = {pk: i for i, pk in enumerate(collection_keys)}

def mine_hard_negatives(query_emb, top_k=HARD_NEG_POOL, n_neg=N_HARD_NEGS, gold_pubkeys=None):
    """For each query, return n_neg doc indices ranked highly by dense but != gold."""
    scores = (query_emb @ coll_emb.T).numpy()
    sorted_idx = np.argsort(-scores, axis=1)[:, :top_k]
    out = []
    for i, gpk in enumerate(gold_pubkeys):
        gold_i = pubkey_to_idx.get(gpk, -1)
        cand = [int(j) for j in sorted_idx[i] if j != gold_i]
        if len(cand) >= n_neg:
            sample = random.sample(cand, n_neg)
        else:
            sample = cand
        out.append(sample)
    return out

In [ ]:
def build_train_triplets(model):
    """Encode all train queries, mine hard negs, return InputExamples (anchor, pos, neg)."""
    examples = []
    for lang in LANGS:
        train  = lang2train[lang]
        texts  = [ex["text"] for ex in train]
        labels = [ex["pubkey"] for ex in train]

        q_emb = encode_e5(model, texts, prefix="query")
        negs  = mine_hard_negatives(q_emb, gold_pubkeys=labels)

        added = 0
        for q_text, gpk, neg_idxs in zip(texts, labels, negs):
            gold_i = pubkey_to_idx.get(gpk)
            if gold_i is None:  # gold not in collection — skip
                continue
            pos_text = collection_texts[gold_i]
            for ni in neg_idxs:
                examples.append(InputExample(
                    texts=[f"query: {q_text}", f"passage: {pos_text}", f"passage: {collection_texts[ni]}"]
                ))
                added += 1
        print(f"  {lang}: +{added} triplets")
    random.shuffle(examples)
    return examples

# 8) Bi-encoder fine-tuning — 2 rounds with iterative hard-neg mining

In [ ]:
def fine_tune_round(model, examples, epochs=FT_EPOCHS, batch_size=FT_BATCH_SIZE, tag="r1"):
    print(f"[{tag}] {len(examples)} triplets, {epochs} epochs, bs={batch_size}")
    loader = DataLoader(examples, shuffle=True, batch_size=batch_size)
    loss = losses.MultipleNegativesRankingLoss(model)
    warmup = int(0.1 * len(loader) * epochs)
    model.fit(
        train_objectives=[(loader, loss)],
        epochs=epochs,
        warmup_steps=warmup,
        use_amp=True,
        show_progress_bar=True,
        output_path=str(MODEL_DIR / f"biencoder_{tag}"),
    )
    return model

In [ ]:
# ROUND 1
print("=" * 60); print("ROUND 1 — mining + training"); print("=" * 60)
r1_examples = build_train_triplets(ft_model)
ft_model = fine_tune_round(ft_model, r1_examples, tag="r1")

# Re-encode collection with FT model
print("\nRe-encoding collection with FT-r1 model...")
coll_emb = encode_e5(ft_model, collection_texts, prefix="passage", batch_size=128)

In [ ]:
# ROUND 2
if N_FT_ROUNDS >= 2:
    print("=" * 60); print("ROUND 2 — re-mining + training"); print("=" * 60)
    r2_examples = build_train_triplets(ft_model)
    ft_model = fine_tune_round(ft_model, r2_examples, tag="r2")

    print("\nRe-encoding collection with FT-r2 model...")
    coll_emb = encode_e5(ft_model, collection_texts, prefix="passage", batch_size=128)

# 9) First-stage retrieval — FT-dense + BM25 RRF


In [ ]:
def tokenize_en(text):
    return re.findall(r"\w+", (text or "").lower())

print("Building BM25 index...")
bm25_corpus = [tokenize_en(t) for t in collection_texts]
bm25 = BM25Okapi(bm25_corpus)
print("Done.")

In [ ]:
def rrf_fusion(score_arrays, k=60):
    fused = np.zeros_like(score_arrays[0], dtype=float)
    for s in score_arrays:
        ranks = np.argsort(np.argsort(-s, axis=1), axis=1)
        fused += 1.0 / (k + ranks + 1)
    return fused

def first_stage(query_orig, model):
    """Dense-only first stage. BM25 was dropped — it consistently hurt MRR@5 across
    all three languages when combined with a fine-tuned dense model via RRF."""
    q_emb = encode_e5(model, query_orig, prefix="query")
    dense = (q_emb @ coll_emb.T).numpy()
    return dense

In [ ]:
def mrr_at_5(scores, gold_pubkeys):
    top5 = np.argsort(-scores, axis=1)[:, :5]
    top5_keys = collection_keys[top5]
    gold = np.asarray(gold_pubkeys)[:, None]
    matches = (top5_keys == gold)
    pos = np.argmax(matches, axis=1)
    pos[~matches.any(axis=1)] = -1
    rr = np.zeros(len(pos), dtype=float)
    rr[pos >= 0] = 1.0 / (pos[pos >= 0] + 1)
    return float(rr.mean())

print("=== First-stage dev MRR@5 ===")
lang2dev_first = {}
for lang in LANGS:
    dev_orig = [ex["text"] for ex in lang2dev[lang]]
    dense = first_stage(dev_orig, ft_model)
    lang2dev_first[lang] = dense
    gold = [ex["pubkey"] for ex in lang2dev[lang]]
    print(f"  {lang}: MRR@5={mrr_at_5(dense, gold):.4f}")

# 10) Cross-encoder reranker fine-tuning

Use translated EN queries vs. paper text. Hard negatives sampled from current FT-dense top-50.
Binary labels: 1 for gold, 0 for hard negative.

In [ ]:
def build_reranker_examples(first_stage_train, n_neg=5, neg_pool=50, skip_top=5):
    """Mine hard negatives from first-stage RRF scores.
    skip_top: skip the very highest-ranked candidates — they may be unlabelled positives
    (false negatives) that would poison training if treated as negatives."""
    examples = []
    for lang in LANGS:
        train    = lang2train[lang]
        train_en = lang2train_en[lang]
        labels   = [ex["pubkey"] for ex in train]
        fs       = first_stage_train[lang]
        top_pool = np.argsort(-fs, axis=1)[:, skip_top:neg_pool]  # skip top-skip_top

        for i, (q_en, gpk) in enumerate(zip(train_en, labels)):
            gold_i = pubkey_to_idx.get(gpk)
            if gold_i is None: continue
            examples.append(InputExample(texts=[q_en, collection_texts_plain[gold_i]], label=1.0))
            negs = [int(j) for j in top_pool[i] if j != gold_i][:n_neg]
            for ni in negs:
                examples.append(InputExample(texts=[q_en, collection_texts_plain[ni]], label=0.0))
    random.shuffle(examples)
    return examples

In [ ]:
# First-stage scores for train queries (needed for hard-neg mining)
print("Computing first-stage train scores for reranker hard-neg mining...")
lang2train_first = {}
for lang in LANGS:
    train_orig = [ex["text"] for ex in lang2train[lang]]
    train_en   = lang2train_en[lang]
    dense = first_stage(train_orig, ft_model)
    lang2train_first[lang] = dense
    print(f"  {lang}: {dense.shape}")

print("\nBuilding reranker training data (CE hard negatives)...")
re_examples = build_reranker_examples(lang2train_first)
print(f"Total reranker examples: {len(re_examples)}")

reranker = CrossEncoder(RERANKER_MODEL, max_length=512, device=DEVICE)
re_loader = DataLoader(re_examples, shuffle=True, batch_size=RERANK_FT_BATCH)
warmup = int(0.1 * len(re_loader) * RERANK_FT_EPOCHS)
reranker.fit(
    train_dataloader=re_loader,
    epochs=RERANK_FT_EPOCHS,
    warmup_steps=warmup,
    optimizer_params={"lr": 1e-5},
    use_amp=True,
    show_progress_bar=True,
    output_path=str(MODEL_DIR / "reranker_ft"),
)

# 11) Cross-encoder rerank → top-20


In [ ]:
def ce_rerank(query_en_texts, first_stage_scores, top_k=FIRST_STAGE_TOPK, batch_qs=8):
    """Rerank top-k candidates per query with the FT cross-encoder.
    Output: scores matrix where reranked positions have CE scores, rest -inf."""
    n_q, n_d = first_stage_scores.shape
    out = np.full((n_q, n_d), -np.inf, dtype=float)
    for start in tqdm(range(0, n_q, batch_qs), desc="CE rerank"):
        end = min(start + batch_qs, n_q)
        pairs, qis, dis = [], [], []
        for i in range(start, end):
            top = np.argsort(-first_stage_scores[i])[:top_k]
            for d in top:
                pairs.append((query_en_texts[i], collection_texts_plain[int(d)]))
                qis.append(i); dis.append(int(d))
        scores = reranker.predict(pairs, batch_size=64, show_progress_bar=False)
        for s, qi, di in zip(scores, qis, dis):
            out[qi, di] = float(s)
        torch.cuda.empty_cache()
    return out

In [ ]:
print("=== Cross-encoder reranked dev MRR@5 ===")
lang2dev_ce = {}
for lang in LANGS:
    print(f"\n--- {lang} ---")
    ce_scores = ce_rerank(lang2dev_en[lang], lang2dev_first[lang])
    lang2dev_ce[lang] = ce_scores
    gold = [ex["pubkey"] for ex in lang2dev[lang]]
    print(f"  CE MRR@5: {mrr_at_5(ce_scores, gold):.4f}")

# Recall@K

In [ ]:
def recall_at_k(scores, gold_pubkeys, k):
    topk = np.argsort(-scores, axis=1)[:, :k]
    topk_keys = collection_keys[topk]
    gold = np.asarray(gold_pubkeys)[:, None]
    hits = (topk_keys == gold).any(axis=1)
    return float(hits.mean())

print(f"{'Stage':<30} {'K':>5}  {'de':>7} {'fr':>7} {'en':>7}")
print("-" * 60)
for stage_name, scores_dict in [
    ("First stage (RRF)",   lang2dev_first),
    ("Cross-encoder",       lang2dev_ce),
]:
    for k in ([5, 20, 100] if stage_name.startswith("First") else [5, 20]):
        row = f"  {stage_name:<28} {k:>5}"
        for lang in LANGS:
            gold = [ex["pubkey"] for ex in lang2dev[lang]]
            v = recall_at_k(scores_dict[lang], gold, k)
            row += f"  {v:.3f}"
        print(row)

In [ ]:
def minmax_norm(x):
    # Use nanmin/nanmax so -inf positions (non-reranked) don't corrupt the range.
    tmp = x.copy().astype(float)
    tmp[~np.isfinite(tmp)] = np.nan
    mn = np.nanmin(tmp, axis=1, keepdims=True)
    mx = np.nanmax(tmp, axis=1, keepdims=True)
    normed = (x - mn) / (mx - mn + 1e-9)
    normed[~np.isfinite(x)] = -np.inf   # restore non-finite positions
    return normed

# Interpolate: CE (primary) + first-stage (secondary nudge inside reranked pool)
# Alpha controls how much first-stage pulls; low value keeps CE as dominant signal.
INTERP_ALPHA = 0.15

print("=== Stage 3: CE + first-stage interpolation (dev) ===")
lang2dev_final = {}
for lang in LANGS:
    ce_n  = minmax_norm(lang2dev_ce[lang].copy())
    fs_n  = minmax_norm(lang2dev_first[lang].copy())
    final = ce_n + INTERP_ALPHA * fs_n
    # Where CE gave -inf (not reranked), set to -inf so they don't compete
    final[lang2dev_ce[lang] == -np.inf] = -np.inf
    lang2dev_final[lang] = final
    gold = [ex["pubkey"] for ex in lang2dev[lang]]
    print(f"  {lang}: MRR@5 = {mrr_at_5(final, gold):.4f}  "
          f"(ce-only = {mrr_at_5(lang2dev_ce[lang], gold):.4f})")

# 12) Dev summary


In [ ]:
rows = []
for lang in LANGS:
    gold = [ex["pubkey"] for ex in lang2dev[lang]]
    rows.append([
        lang,
        mrr_at_5(lang2dev_first[lang], gold),
        mrr_at_5(lang2dev_ce[lang],    gold),
        mrr_at_5(lang2dev_final[lang], gold),
    ])
df = pd.DataFrame(rows, columns=["lang", "first-stage (ft+bm25 RRF)", "+ CE rerank", "+ interp"])
print(df.to_string(index=False))

In [ ]:
print("=== Saving best dev predictions (CE + interpolation) ===")
for lang in LANGS:
    scores = lang2dev_final[lang]
    top5 = np.argsort(-scores, axis=1)[:, :5]
    top5_keys = collection_keys[top5]
    df = lang2dev[lang].to_pandas()[["index"]]
    df["preds"] = top5_keys.tolist()
    out_path = OUT_DIR / f"dev_predictions_{lang}.tsv"
    df.to_csv(out_path, index=False, sep="\t")
    print(f"  Saved {out_path} ({len(df)} queries)")

subprocess.run(["zip", "-j", str(OUT_DIR / "dev_predictions.zip")] + [str(OUT_DIR / f"dev_predictions_{l}.tsv") for l in LANGS])
print("dev_predictions.zip ready.")

In [ ]:
def run_full_pipeline(query_orig, query_en):
    dense = first_stage(query_orig, ft_model)
    ce_scores = ce_rerank(query_en, dense)
    ce_n = minmax_norm(ce_scores.copy())
    fs_n = minmax_norm(dense.copy())
    final = ce_n + INTERP_ALPHA * fs_n
    final[ce_scores == -np.inf] = -np.inf
    return final

print("=== Test inference + export ===")
for lang in LANGS:
    print(f"\n--- {lang} test ---")
    test_orig = [ex["text"] for ex in lang2test[lang]]
    test_en   = lang2test_en[lang]
    final     = run_full_pipeline(test_orig, test_en)

    top5 = np.argsort(-final, axis=1)[:, :5]
    top5_keys = collection_keys[top5]

    df = lang2test[lang].to_pandas()[["index"]]
    df["preds"] = top5_keys.tolist()
    out_path = OUT_DIR / f"predictions_{lang}.tsv"
    df.to_csv(out_path, index=False, sep="\t")
    print(f"  Saved {out_path} ({len(df)} queries)")

# Zip
zip_path = OUT_DIR / "test_predictions.zip"
subprocess.run(["zip", "-j", "-r", str(zip_path)] + [str(OUT_DIR / f"predictions_{l}.tsv") for l in LANGS])
print(f"\nReady for submission: {zip_path}")

In [ ]:
def run_full_pipeline(query_orig, query_en, tag):
    rrf, _, _ = first_stage(query_orig, query_en, ft_model)
    ce_scores = ce_rerank(query_en, rrf)
    # final     = llm_rerank_split(query_en, ce_scores, tag=tag)
    return ce_scores

print("=== Test inference + export ===")
for lang in LANGS:
    print(f"\n--- {lang} test ---")
    test_orig = [ex["text"] for ex in lang2test[lang]]
    test_en   = lang2test_en[lang]
    final     = run_full_pipeline(test_orig, test_en, tag=f"test-{lang}")

    top5 = np.argsort(-final, axis=1)[:, :5]
    top5_keys = collection_keys[top5]

    df = lang2test[lang].to_pandas()[["index"]]
    df["preds"] = top5_keys.tolist()
    out_path = OUT_DIR / f"predictions_{lang}.tsv"
    df.to_csv(out_path, index=False, sep="\t")
    print(f"  Saved {out_path} ({len(df)} queries)")

# Zip
zip_path = OUT_DIR / "predictions.zip"
subprocess.run(["zip", "-j", "-r", str(zip_path)] + [str(OUT_DIR / f"predictions_{l}.tsv") for l in LANGS])
print(f"\nReady for submission: {zip_path}")